# NASA source-to-target verification — corrected v2

The v2 revision fixes the mapping alias `R5E_Interval_Cycle_Position` → `Interval_Position` for RW cycles. It also reports *exact*, *proven newline-equivalent*, and *unexplained* Git/manifest SHA-256 outcomes separately; it does not change published files or claim that a newline-equivalent file has an exact SHA-256 match.

**Scope:** Preserved study-format NASA CSVs → reconstructed capacities and mappings → frozen Step 11A; archived Step 11N/11O numerical verification. NASA native `.mat` conversion and historical model refits remain unverified.

**How to run:** Runtime → Restart and run all. A ZIP is saved even if checks fail.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,hashlib,json,zipfile,shutil,subprocess,csv,datetime,base64,time

# Adjust only this path if your archive moves.
COLLAB_ZIP=Path('/content/drive/MyDrive/Battery_SOH_Project/raw_data/collab.zip')
assert COLLAB_ZIP.is_file(), f'Archive not found: {COLLAB_ZIP}'
REPO_URL='https://github.com/batteryP30/battery_paper.git'
RELEASE_TAG='v0.9.0'
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORK=Path('/content/nasa_source_verification')
REPO=WORK/'battery_paper'
TMP=WORK/'source_csv'
OUTPUT=WORK/'results'/RUN_ID
DRIVE_OUT=Path('/content/drive/MyDrive/Battery_SOH_Project/reviewer_response/nasa_source_verification')
for folder in (WORK,TMP,OUTPUT,DRIVE_OUT): folder.mkdir(parents=True,exist_ok=True)
print('Experiment:',RUN_ID,'Archive:',COLLAB_ZIP)

In [ ]:
# Clone immutable GitHub release to ensure a known source-code version.
# This only clones files into Colab; it does not write or commit anything to GitHub.
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','--branch',RELEASE_TAG,REPO_URL,str(REPO)],check=True)
actual_commit=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print('Repository tag:', RELEASE_TAG, '\nExact commit:', actual_commit)
# Colab usually supplies these dependencies; install the project's actual dependencies if missing.
subprocess.run([sys.executable,'-m','pip','-q','install','-e',str(REPO/'battery_soh_reproducibility')],check=True)

## Frozen Step 11A evidence bundled in this notebook

The following cell restores the CSV audit values and verification programs from the previously recovered Step 11A archive. It does **not** generate synthetic battery records or supply raw NASA measurements.

In [ ]:
EMBEDDED_AUDIT_SHA256='bfd8544ac6567ea99b64e9db9375ee8b6530459b670a5dbf62ec4cd751db2c72'
EMBEDDED_AUDIT_B64="""
...
"""
payload=base64.b64decode(''.join(EMBEDDED_AUDIT_B64.split()))
assert hashlib.sha256(payload).hexdigest()==EMBEDDED_AUDIT_SHA256
from io import BytesIO
with zipfile.ZipFile(BytesIO(payload)) as z:
    names=z.namelist()
    assert all(not x.startswith('/') and '..' not in Path(x).parts for x in names)
    z.extractall(REPO)
print('Restored',len(names),'frozen audit/checking files to temporary Colab checkout.')
worker=REPO/'tools/nasa_worker.py'
assert worker.is_file()
# Do not save the embedded payload as a raw-data release asset.
del EMBEDDED_AUDIT_B64, payload


In [ ]:
# Preflight: verify all eight expected study CSVs are present before processing.
NASA_IDS=['B0005','B0006','B0007','B0018','RW3','RW4','RW5','RW6']
expected={b:f'{b}/{b}.csv' for b in NASA_IDS}
with zipfile.ZipFile(COLLAB_ZIP) as archive:
    info={i.filename:i for i in archive.infolist() if not i.is_dir()}
    missing=[name for name in expected.values() if name not in info]
    assert not missing, f'Missing expected archive paths: {missing}'
    for b,path in expected.items():
        ent=info[path]
        print(f'{b:6s}  {ent.file_size:>12,} bytes  {path}')
print('PASS: eight required study CSV exports located; native MATLAB files not present in inventory.')

In [ ]:
# Each battery runs in a NEW Python process so RAM is returned before the next RW CSV.
# Output ZIP contains derived target/event records and per-source hashes, NEVER the raw CSVs.
reports=[]
source_inventory=[]
for battery in NASA_IDS:
    member=expected[battery]
    dest=TMP/f'{battery}.csv'
    print(f'\n===== {battery}: extracting and verifying =====',flush=True)
    # Stream one archive member; never extract the full 2.72-GB archive.
    hash_source=hashlib.sha256()
    count=0
    with zipfile.ZipFile(COLLAB_ZIP) as archive:
        with archive.open(member) as src, open(dest,'wb') as dst:
            while True:
                chunk=src.read(8*1024*1024)
                if not chunk: break
                dst.write(chunk)
                hash_source.update(chunk)
                count+=len(chunk)
    source_inventory.append({'battery_id':battery,'archive_member':member,
                            'size_bytes':count,'sha256':hash_source.hexdigest(),
                            'source_type':'preserved study-format CSV export',
                            'native_mat_conversion_verified':False})
    command=[sys.executable,str(worker), '--repo',str(REPO),
             '--audit',str(REPO/'research_release/editor_traceability'),
             '--battery',battery, '--source',str(dest),'--output-dir',str(OUTPUT)]
    try:
        run=subprocess.run(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                           text=True,timeout=3600)
        log=run.stdout
        rc=run.returncode
    except subprocess.TimeoutExpired as err:
        log=str(err)
        rc=124
    finally:
        dest.unlink(missing_ok=True)
    (OUTPUT/f'{battery}_worker.log').write_text(log+'\n',encoding='utf-8')
    checkfile=OUTPUT/f'{battery}_check.json'
    result=json.loads(checkfile.read_text()) if checkfile.exists() else {
        'battery':battery,'status':'FAIL','reason':f'Process did not finish; returncode={rc}'}
    result['source_sha256']=hash_source.hexdigest()
    result['source_size_bytes']=count
    result['worker_returncode']=rc
    reports.append(result)
    print('Status:',result['status'],'|',result.get('reason',''),flush=True)
    if result['status']=='PASS':
        print('Reconstructed reference capacity (Ah):',result.get('reference_Ah'),flush=True)
    else:
        print('Check',OUTPUT/f'{battery}_worker.log','for the precise discrepancy.',flush=True)
# Persist summary even if a particular battery fails.
import pandas as pd
pd.DataFrame(source_inventory).to_csv(OUTPUT/'NASA_Source_Manifest_SHA256.csv',index=False)
(OUTPUT/'NASA_Verification_Summary.json').write_text(json.dumps(reports,indent=2,default=str)+'\n',encoding='utf-8')
print('\nSource CSV comparisons:',sum(r['status']=='PASS' for r in reports),'/',len(reports),'passed')

In [ ]:
# Independent checks. Do NOT rewrite or bypass the published checksum manifest.
# A Windows-generated v0.9.0 manifest may refer to CRLF bytes that Git stored
# as LF-normalized text. Identify ONLY byte-proven newline equivalences; log any
# other discrepancy as a genuine integrity failure.
import hashlib, importlib.util

manifest=REPO/'research_release/MANIFEST.sha256'
exact_matches=[]
eol_equivalent=[]
unexplained=[]
for line in manifest.read_text(encoding='utf-8').splitlines():
    if not line.strip(): continue
    expected,relative=line.split('  ',1)
    file=REPO/relative
    if not file.is_file():
        unexplained.append({'path':relative,'classification':'MISSING','expected':expected,'actual':None})
        continue
    raw=file.read_bytes()
    actual=hashlib.sha256(raw).hexdigest()
    if actual==expected:
        exact_matches.append({'path':relative,'classification':'EXACT','expected':expected,'actual':actual})
        continue
    # No guessing: only classify as EOL-equivalent when SHA256 of the
    # reverse line-ending transform EXACTLY equals the published checksum.
    if file.suffix.lower() in ('.csv','.md','.txt','.json','.py','.toml','.yaml','.yml') and b'\x00' not in raw:
        variants={
           'CHECKOUT_LF_VS_FROZEN_CRLF': raw.replace(b'\r\n', b'\n').replace(b'\n',b'\r\n'),
           'CHECKOUT_CRLF_VS_FROZEN_LF': raw.replace(b'\r\n',b'\n'),
        }
        match=next((typ for typ,candidate in variants.items()
                    if candidate!=raw and hashlib.sha256(candidate).hexdigest()==expected),None)
        if match:
            eol_equivalent.append({'path':relative,'classification':match,'expected':expected,'actual':actual})
            continue
    unexplained.append({'path':relative,'classification':'UNEXPLAINED','expected':expected,'actual':actual})

import pandas as pd
pd.DataFrame(exact_matches+eol_equivalent+unexplained,
    columns=['path','classification','expected','actual']).to_csv(
    OUTPUT/'Release_Manifest_Diagnostics.csv',index=False)
print('\nPublished manifest versus Git checkout:')
print('  exact file hashes:',len(exact_matches))
print('  EOL-only hash differences, reverse-transform proven:',len(eol_equivalent))
print('  unexplained mismatches:',len(unexplained))
for item in (eol_equivalent+unexplained)[:12]:
    print(' ',item['classification'],item['path'])

# Invoke the originally released verifier. Preserve its failure in the log.
verifier=REPO/'research_release/tools/verify_release.py'
run=subprocess.run([sys.executable,str(verifier)],cwd=REPO,
      text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(OUTPUT/'verify_release_original.log').write_text(run.stdout,encoding='utf-8')
print('\n==== verify_release.py (unmodified): exit',run.returncode,'====\n'+run.stdout[-1700:])

# Only if every manifest discrepancy is PROVEN to be a newline translation,
# independently run the same frozen numerical checks without falsely claiming
# an exact-byte manifest PASS. The original verifier code itself is not edited.
metrics_ok=False
metrics_mode='NOT_RUN'
if run.returncode==0:
    metrics_ok=True
    metrics_mode='EXACT_MANIFEST_AND_NUMERICAL_CHECKS'
elif not unexplained and len(eol_equivalent)>0:
    spec=importlib.util.spec_from_file_location('frozen_release_checker',str(verifier))
    vr=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(vr)
    vr.check_manifest=lambda:print('CHECKSUM STATUS: exact published hash differs ONLY by proven line-ending normalization; original check failed and remains documented.')
    try:
        vr.main()   # Original numerical and structural checks, with integrity diagnosed separately
        metrics_ok=True
        metrics_mode='NUMERICAL_CHECKS_PASS_WITH_PROVEN_EOL_MANIFEST_DIFFERENCES'
    except Exception as e:
        print('FAIL: archived numerical verification:',repr(e))
else:
    print('FAIL: unexplained manifest mismatch; archived numerical checks cannot be certified here.')

(OUTPUT/'Archive_Checks_Classification.json').write_text(json.dumps({
    'manifest_exact_files':len(exact_matches),
    'manifest_eol_equivalent_files':len(eol_equivalent),
    'manifest_unexplained_files':len(unexplained),
    'original_verifier_exit_code':run.returncode,
    'numerical_validation_passed':metrics_ok,
    'numerical_validation_mode':metrics_mode,
    'warning':'Line-ending equivalence is not an exact-byte SHA256 match. Fix Git-tracked release manifest in next deposit; never describe v0.9.0 as byte-identical if differences exist.'
},indent=2)+'\n',encoding='utf-8')

# Separately verify the archived capacity-to-fold-to-OOF chain.
chain_script=REPO/'research_release/tools/verify_editor_chain.py'
chain=subprocess.run([sys.executable,str(chain_script),'--repo-root',str(REPO),
   '--json-output',str(OUTPUT/'Editor_Archived_Chain_Report.json')],cwd=REPO,
   text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(OUTPUT/'verify_editor_chain.log').write_text(chain.stdout,encoding='utf-8')
print('\n==== verify_editor_chain.py: exit',chain.returncode,'====\n'+chain.stdout[-2200:])
downstream=[
 {'script':'verify_release.py','exit_code':run.returncode,'numerical_validation_passed':metrics_ok,'mode':metrics_mode},
 {'script':'verify_editor_chain.py','exit_code':chain.returncode},
]
(OUTPUT/'Verification_Environment.json').write_text(json.dumps({
    'run_utc':RUN_ID,'github_repository':REPO_URL,'github_tag':RELEASE_TAG,
    'github_commit':actual_commit,'archive_filename':COLLAB_ZIP.name,
    'audit_embedded_sha256':EMBEDDED_AUDIT_SHA256,
    'per_battery_status':{r['battery']:r['status'] for r in reports},
    'downstream_checks':downstream,
    'scope':'NASA study CSV exports to archived Step11A/11N/11O; NO native MATLAB conversion or model refit',
},indent=2)+'\n',encoding='utf-8')


In [ ]:
# Save provenance evidence even if checks fail; never distribute raw CSVs.
readme_lines = [
  'NASA study CSV exports -> archived Step11A reconstruction -> preserved prediction verification.',
  'Raw battery measurements are NOT distributed in this ZIP.',
  'SHA-256 identifies the exact study-format CSVs processed.',
  'The released v0.9.0 reconstruction code was used; no models were refitted.',
  'Native NASA MATLAB -> study-format CSV conversion is NOT verified.',
  'Review Release_Manifest_Diagnostics.csv; any Git newline change is NOT a raw-byte SHA256 match.',
  'Review per-battery logs and Archive_Checks_Classification.json before asserting PASS.',
]
(OUTPUT/'READ_THIS_FIRST.txt').write_text('\n'.join(readme_lines)+'\n',encoding='utf-8')
output_zip=DRIVE_OUT/f'NASA_Source_to_Archived_Chain_v2_{RUN_ID}.zip'
with zipfile.ZipFile(output_zip,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for file in sorted(OUTPUT.rglob('*')):
        if file.is_file():z.write(file,file.relative_to(OUTPUT))
print('\nSaved:',output_zip)
print('Result archive size:',output_zip.stat().st_size,'bytes')
source_ok=all(r['status']=='PASS' and r.get('worker_returncode')==0 for r in reports)
print('SOURCE CHECK:',sum(r['status']=='PASS' for r in reports),'/',len(reports),'PASS')
print('ARCHIVED METRICS:',metrics_mode)
print('ARCHIVED CHAIN:','PASS' if chain.returncode==0 else 'FAIL')
if not (source_ok and metrics_ok and chain.returncode==0):
    raise RuntimeError('Some checks failed; saved ZIP contains detailed evidence. Do not modify frozen CSVs or published manifest.')
if eol_equivalent:
    print('PASS: eight study-CSV-to-Step11A source checks and archived numerical chain. WARNING: v0.9.0 checksum portability requires a future release correction; see report.')
else:
    print('PASS: eight study-CSV-to-Step11A source checks, original manifest and archived numerical chain.')
print('NOT VERIFIED: NASA native .mat conversion, CALCE source reconstruction, or historical full model refits.')
